# 04 — Build master dataset & candidate feature pool (Medellín, PM2.5 target)

Mirrors Bogotá's `05_build_master_dataset.ipynb`, re-centered on Medellín. Differences:

- Meteorology (wind speed/dir, temp, RH, pressure) comes from ERA5 here, not station
  gauges (see notebook 03) -- Medellín has no comparably good local coverage in what we
  have locally.
- `no` is already a column in `medellin_network_daily_clean.parquet` (built directly
  from `Hourmean/NO_hourmean.csv` in notebook 01) -- no separate NO-extraction notebook
  needed, unlike Bogotá.
- No v6 ERA5 extras (tcwv/sensible_heat_flux/tke_proxy/w700) yet -- Medellín's ERA5 pull
  only covers the core variable set for now (see notebook 03).
- `nox = no + no2` added as a vehicle-emissions proxy, same reasoning as the Bogotá
  pipeline's own v8 addition.


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("/path/to/project")
sys.path.insert(0, str(PROJECT_ROOT / "Scripts"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from common.external_data import FIRE_SOURCE_REGIONS, bearing_from_bogota, MEDELLIN_LAT, MEDELLIN_LON

DATA_DIR = PROJECT_ROOT / "Data" / "processed" / "colombia" / "medellin"
FIG_DIR = PROJECT_ROOT / "Scripts" / "colombia" / "medellin" / "figures"

network = pd.read_parquet(DATA_DIR / "medellin_network_daily_clean.parquet")
era5 = pd.read_parquet(DATA_DIR / "medellin_era5_daily.parquet")
hotspots = pd.read_parquet(DATA_DIR / "medellin_hotspots_daily.parquet")
w700 = pd.read_parquet(DATA_DIR / "medellin_w700_daily.parquet")
antioquia = pd.read_parquet(DATA_DIR / "medellin_antioquia_network_daily.parquet")
antioquia_new = pd.read_parquet(DATA_DIR / "medellin_antioquia_network_new_daily.parquet")  # CO/SO2/etc. batch, 2026-09-23
print("antioquia_new (CO/SO2/etc.):", antioquia_new.shape, list(antioquia_new.columns))
lts = pd.read_parquet(DATA_DIR / "medellin_lts_daily.parquet")
station_meteo_check = pd.read_parquet(DATA_DIR / "medellin_meteo_daily_clean.parquet")
print("station meteo:", station_meteo_check.shape, station_meteo_check["date"].min(), station_meteo_check["date"].max())

print("network:", network.shape, network['date'].min(), network['date'].max())
print("era5:", era5.shape, era5['date'].min(), era5['date'].max())
print("hotspots:", hotspots.shape)
print("pm25 non-null:", network['pm25'].notna().sum())


antioquia_new (CO/SO2/etc.): (5331, 9) ['date', 'co', 'so2', 'no_sisaire', 'nox_reported', 'pst', 'radiation_station', 'temp_10cm', 'precip_station']
station meteo: (2919, 5) 2013-01-01 00:00:00 2020-12-31 00:00:00
network: (3652, 6) 2013-01-01 00:00:00 2022-12-31 00:00:00
era5: (2923, 15) 2014-12-31 00:00:00 2022-12-31 00:00:00
hotspots: (4017, 12)
pm25 non-null: 2874


## Merge

Three-source cascade, primary-with-calibrated-fallback at each level (same technique as
the station-vs-ERA5 meteorology fix earlier -- linear calibration on the overlap period
before filling gaps, never a raw splice, to avoid another Simpson's-paradox-style
level-shift artifact):

1. **`pm25`/`pm10`/`no2`/`wind_speed`/`wind_dir`**: primary source is now the
   Aburra-Valley-filtered, properly QC'd Antioquia SISAIRE network (notebook 00 --
   Chauvenet + >=75% completeness, 43-55 stations, 2008-2025), calibrated-fallback is
   whatever the previous primary source was for that variable (`Hourmean` for the
   pollutants, the existing station+ERA5 hybrid for wind) to fill occasional gaps within
   the Antioquia record. This replaces a shorter (2015-2022), undocumented-provenance
   `Hourmean` network mean with a transparent, much longer, larger-network-averaged
   series for the pollutants, and extends wind past the previous 2020 cutoff.
2. **`temp_10m`/`rh_10m`**: new standalone columns, straight from the Antioquia network
   (no prior 10m-height source existed to blend with) -- `temp_2m`/`rh_2m` (from the
   station+ERA5 hybrid) are unchanged and kept alongside.
3. **`pressure` dropped from the Antioquia source entirely** -- most of that network's
   barometric sensors are miscalibrated (per-station means ranging 551-872 hPa for a
   city where physically-correct is ~800-870 hPa; only 4/52 stations were plausible).
   ERA5's `pressure` remains the sole source, as before.
4. `no`/`o3` are unaffected -- still from the `Hourmean`-sourced `network` table (O3
   stays MDA8, not rebuilt from Antioquia's plain daily-mean `O3` column, to avoid
   silently blending two different ozone metrics -- see notebook 00's markdown).


In [2]:
station_meteo = pd.read_parquet(DATA_DIR / "medellin_meteo_daily_clean.parquet")
station_meteo = station_meteo.rename(columns={
    'temp_2m': 'temp_2m_station', 'rh_2m': 'rh_2m_station',
    'wind_speed': 'wind_speed_station', 'wind_dir': 'wind_dir_station',
})

antioquia_r = antioquia.rename(columns={
    'pm25': 'pm25_antioquia', 'pm10': 'pm10_antioquia', 'no2': 'no2_antioquia',
    'wind_speed': 'wind_speed_antioquia', 'wind_dir': 'wind_dir_antioquia',
})

df = (network.rename(columns={'pm25': 'pm25_hourmean', 'pm10': 'pm10_hourmean', 'no2': 'no2_hourmean'})
             .merge(era5, on='date', how='left')
             .merge(hotspots, on='date', how='left')
             .merge(station_meteo, on='date', how='left')
             .merge(w700, on='date', how='left')
             .merge(lts, on='date', how='left')
             .merge(antioquia_r, on='date', how='left')
             .merge(antioquia_new, on='date', how='left'))  # CO/SO2/etc. batch, 2026-09-23
df = df.sort_values('date').reset_index(drop=True)

df = df.rename(columns={
    'wind_speed': 'wind_speed_era5', 'wind_dir': 'wind_dir_era5',
    'temp_2m': 'temp_2m_era5', 'rh_2m': 'rh_2m_era5',
})

from sklearn.linear_model import LinearRegression

def calibrate_and_fill(data, primary_col, fallback_col, label):
    overlap = data[[primary_col, fallback_col]].dropna()
    lr = LinearRegression().fit(overlap[[fallback_col]], overlap[primary_col])
    r2 = lr.score(overlap[[fallback_col]], overlap[primary_col])
    print(f"{label}: calibration {fallback_col}->{primary_col} scale  slope={lr.coef_[0]:.3f}  "
          f"intercept={lr.intercept_:.3f}  overlap_R2={r2:.3f}  (n={len(overlap)})")
    fallback_calibrated = lr.predict(data[[fallback_col]].fillna(0))
    fallback_calibrated = pd.Series(fallback_calibrated, index=data.index).where(data[fallback_col].notna())
    return data[primary_col].combine_first(fallback_calibrated)

# Station meteo -> ERA5 calibration (unchanged from the earlier fix)
for var in ['wind_speed', 'temp_2m', 'rh_2m']:
    df[f'{var}_hybrid_station_era5'] = calibrate_and_fill(df, f'{var}_station', f'{var}_era5', var)
df['wind_dir_hybrid_station_era5'] = df['wind_dir_station'].combine_first(df['wind_dir_era5'])

# Antioquia (primary) -> previous-hybrid-or-Hourmean (fallback) calibration
df['pm25'] = calibrate_and_fill(df, 'pm25_antioquia', 'pm25_hourmean', 'pm25')
df['pm10'] = calibrate_and_fill(df, 'pm10_antioquia', 'pm10_hourmean', 'pm10')
df['no2'] = calibrate_and_fill(df, 'no2_antioquia', 'no2_hourmean', 'no2')
df['wind_speed'] = calibrate_and_fill(df, 'wind_speed_antioquia', 'wind_speed_hybrid_station_era5', 'wind_speed')
df['wind_dir'] = df['wind_dir_antioquia'].combine_first(df['wind_dir_hybrid_station_era5'])
df['temp_2m'] = df['temp_2m_hybrid_station_era5']
df['rh_2m'] = df['rh_2m_hybrid_station_era5']
# temp_10m/rh_10m come straight from antioquia_r, already correctly named -- no action needed.

print(df.shape)
for v in ['pm25', 'pm10', 'no2', 'wind_speed', 'wind_dir', 'temp_10m', 'rh_10m']:
    print(f"{v}: total non-null={df[v].notna().sum()}")


wind_speed: calibration wind_speed_era5->wind_speed_station scale  slope=0.843  intercept=0.781  overlap_R2=0.494  (n=2193)
temp_2m: calibration temp_2m_era5->temp_2m_station scale  slope=1.196  intercept=-1.446  overlap_R2=0.767  (n=2193)
rh_2m: calibration rh_2m_era5->rh_2m_station scale  slope=0.857  intercept=-3.812  overlap_R2=0.727  (n=2193)
pm25: calibration pm25_hourmean->pm25_antioquia scale  slope=0.761  intercept=2.336  overlap_R2=0.752  (n=1736)
pm10: calibration pm10_hourmean->pm10_antioquia scale  slope=0.868  intercept=6.739  overlap_R2=0.901  (n=1426)
no2: calibration no2_hourmean->no2_antioquia scale  slope=0.262  intercept=29.768  overlap_R2=0.077  (n=1771)
wind_speed: calibration wind_speed_hybrid_station_era5->wind_speed_antioquia scale  slope=0.460  intercept=0.654  overlap_R2=0.177  (n=2898)
(3652, 64)
pm25: total non-null=3510
pm10: total non-null=2445
no2: total non-null=3615
wind_speed: total non-null=3652
wind_dir: total non-null=3652
temp_10m: total non-null=

## Engineered features

Same construction as Bogotá's pipeline: wind dir sin/cos, calendar terms, ventilation
coefficient, wind-transport-conditioned hotspots (point regions + compass sectors,
re-centered on Medellín), NO2/ventilation stagnation interaction, secondary-PM proxy,
and NOx as an additional vehicle-emissions proxy.


In [3]:
df['wind_dir_sin'] = np.sin(np.deg2rad(df['wind_dir']))
df['wind_dir_cos'] = np.cos(np.deg2rad(df['wind_dir']))
doy = df['date'].dt.dayofyear
df['doy_sin'] = np.sin(2 * np.pi * doy / 365.25)
df['doy_cos'] = np.cos(2 * np.pi * doy / 365.25)
month = df['date'].dt.month
df['month_sin'] = np.sin(2 * np.pi * month / 12)
df['month_cos'] = np.cos(2 * np.pi * month / 12)
df['day_of_week'] = df['date'].dt.dayofweek
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)

df['ventilation_station'] = df['blh_mean'] * df['wind_speed']

SECTOR_BEARINGS = {name: i * 45.0 for i, name in enumerate(["N", "NE", "E", "SE", "S", "SW", "W", "NW"])}
point_region_cols = [c for c in df.columns if c.startswith('hotspots_') and not c.startswith('hotspots_sector_')]
sector_cols = [c for c in df.columns if c.startswith('hotspots_sector_')]

for region_col in point_region_cols:
    region_name = region_col.replace('hotspots_', '')
    bearing = bearing_from_bogota(
        FIRE_SOURCE_REGIONS[region_name]['lat'], FIRE_SOURCE_REGIONS[region_name]['lon'],
        origin_lat=MEDELLIN_LAT, origin_lon=MEDELLIN_LON,
    )
    alignment = np.cos(np.deg2rad(df['wind_dir'] - bearing)).clip(lower=0)
    df[f'{region_col}_transport'] = df[region_col] * alignment
for sector_col in sector_cols:
    sector_name = sector_col.replace('hotspots_sector_', '')
    bearing = SECTOR_BEARINGS[sector_name]
    alignment = np.cos(np.deg2rad(df['wind_dir'] - bearing)).clip(lower=0)
    df[f'{sector_col}_transport'] = df[sector_col] * alignment

df['no2_stagnation'] = df['no2'] / (1 + df['ventilation_station'])
df['ventilation_min'] = df['blh_min'] * df['wind_speed']
df['fog_stagnation'] = df['rh_2m'] / (1 + df['wind_speed'])
df['dry_stagnation'] = 1 / ((1 + df['wind_speed']) * (1 + df['precip_total_mm']))
df['secondary_pm_proxy'] = df['no2'] * df['radiation_mean']
df['nox'] = df['no'] + df['no2']


In [4]:
LAG_DAYS = [1, 2, 3]
hotspot_region_cols = [c for c in df.columns if c.startswith('hotspots_')]

LAG_VARS = [
    'co', 'so2', 'no_sisaire', 'radiation_station', 'temp_10cm', 'precip_station',  # new Antioquia batch, 2026-09-23 (pst excluded: 0 qualifying station-years; nox_reported excluded: only 2013-2015)
        'pm10', 'no2', 'no', 'nox', 'o3', 'wind_speed', 'temp_2m', 'rh_2m', 'pressure',
    'blh_mean', 'blh_morning_mean', 'radiation_mean', 'precip_total_mm',
    'ventilation_station', 'no2_stagnation',
    'ventilation_min', 'fog_stagnation', 'dry_stagnation', 'secondary_pm_proxy',
    'tcwv', 'sensible_heat_flux', 'tke_proxy', 'w700', 'lts', 'lts_morning',
    'temp_10m', 'rh_10m',
] + hotspot_region_cols
df = df.set_index('date')
for var in LAG_VARS:
    if var not in df.columns:
        continue
    for lag in LAG_DAYS:
        df[f'{var}_lag{lag}'] = df[var].shift(lag)

df['pm25_lag1'] = df['pm25'].shift(1)
df = df.reset_index()

candidate_cols = [c for c in df.columns if c not in ('date', 'pm25')]
print(f"{len(candidate_cols)} candidate predictor columns")


254 candidate predictor columns


/var/folders/30/gtfddbxx5z3f_hc8v4br4pkh0000gn/T/ipykernel_53342/571974551.py:18: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'{var}_lag{lag}'] = df[var].shift(lag)
/var/folders/30/gtfddbxx5z3f_hc8v4br4pkh0000gn/T/ipykernel_53342/571974551.py:18: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'{var}_lag{lag}'] = df[var].shift(lag)
/var/folders/30/gtfddbxx5z3f_hc8v4br4pkh0000gn/T/ipykernel_53342/571974551.py:18: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many

## Verification: same sanity checks as Bogotá

Ventilation should anti-correlate with PM2.5; NO/NOx should correlate positively
(vehicle-emissions proxies).


In [5]:
for col in ['ventilation_station', 'blh_mean', 'wind_speed', 'no', 'nox', 'no2', 'secondary_pm_proxy']:
    if col in df.columns:
        corr = df[['pm25', col]].dropna().corr().iloc[0, 1]
        n = df[['pm25', col]].dropna().shape[0]
        print(f"corr(pm25, {col}) = {corr:.3f}  (n={n})")


corr(pm25, ventilation_station) = 0.051  (n=2896)
corr(pm25, blh_mean) = 0.084  (n=2896)
corr(pm25, wind_speed) = 0.085  (n=3510)
corr(pm25, no) = -0.101  (n=2844)
corr(pm25, nox) = 0.120  (n=2844)
corr(pm25, no2) = 0.129  (n=3503)
corr(pm25, secondary_pm_proxy) = 0.298  (n=2896)


## Save master dataset

In [6]:
out_path = DATA_DIR / "medellin_master_dataset.parquet"
df.to_parquet(out_path, index=False)
print("Saved:", out_path, df.shape)
print("Rows with non-null target (pm25):", df['pm25'].notna().sum())


Saved: /path/to/project/Data/processed/colombia/medellin/medellin_master_dataset.parquet (3652, 256)
Rows with non-null target (pm25): 3510
